# Car vs Truck Modeling

This notebook is the modeling-focused part of the project report. It lightly covers the project context, then documents the baseline and proposed deep-learning models used for the binary image-classification task `car` vs `truck`.

The main goal is to compare one simple baseline (`SimpleCNN`) with two stronger transfer-learning approaches (`ResNet18` and `Swin-Tiny`). The notebook is written to run top to bottom. Its starts with the data loading and some preprocessing (not much is needed because the used CIFAR-10 Dataset is normalized and labeled to some extend) and the resulting datasets are used for the mentionned models to test and train. The model outputs then get evaluated with basic metrics. To get a better understanding of the models behavior, we using LIME to make parts of an example image visible that are influencing the models decision.

# Metadata

```yaml
project_title: "Image Classification between Car and Truck"
dataset_name: "CIFAR-10"
task_type: "binary_image_classification"
methods:
  - simple_cnn
  - resnet18_transfer_learning
  - swin_tiny_transfer_learning
evaluation_metrics:
  - accuracy
  - cross_entropy_loss
team_members:
  - name: Artur Schneider
    student_id:
    role: Data Preprocessing, Project Manager
  - name:
    student_id:
    role:
  - name:
    student_id:
    role:
  - name:
    student_id:
    role:  

submission_date: 
```


## AI Usage Declaration

| Item | Response |
|---|---|
| Did you use generative AI tools? | Yes /  |
| Which tools? | Microsoft Copilot  |
| For what purpose? | brainstorming and coding help |
| Which parts were AI-assisted? | Data Preprocessing |
| What did you verify yourself? | Everything |

### Short declaration
Copilot was used for Braistorming and elaborate on manual written code with giving feedback. Results were checked by comparing it with the taught content of the course (especially Laboratory). Intermediate Steps were visualized and printed out as much as possible to ensure plausibility and explainability.

## 1. Problem Definition and Motivation

## Contribution Metadata
Author(s): Artur Schneider

Section: Problem Definition and Motivation  
Method: [binary-classification, image-recognition]  
SHORT Description of contribution: Framed the modeling task and its motivation.

Difficulty: BASIC

The task is to classify an input image into one of two categories: `car` or `truck`. This is a compact supervised computer-vision problem that is suitable for comparing a simple convolutional baseline against stronger pretrained architectures. The task is relevant because it measures how well different modeling choices separate visually similar road-vehicle classes while keeping the label space small enough for clear analysis. In real-life it can be practical for traffic analysis. 


## 2. Dataset Description

## Contribution Metadata
Author(s): Artur Schneider

Section: Dataset Description  
Method: [cifar10-subset, binary-labels]  
SHORT Description of contribution: Summarized the dataset assumptions needed by the modeling notebook.

Difficulty: BASIC

This notebook expects two ready-to-use PyTorch datasets named `train_dataset` and `test_dataset`. Each dataset item must be a pair `(image_tensor, label)`, where the image has shape `(3, H, W)` and the label is encoded as `0` for `car` and `1` for `truck`. For this binary task we are using the CIFAR-10 Dataset by filtering the original automobile and truck classes. Its a well-known dataset with 10 classes (each class has 10000 train images and 2000 test images) that was researched a lot and providing data in good quality. We decided to use this for ensuring very good model outputs and save time.


## 3. Data Exploration and Preprocessing

## Contribution Metadata
Author(s):  
Section: Data Exploration and Preprocessing  
Method: [dataset-checks, dataloaders, normalization]  
SHORT Description of contribution: Documented the minimal preprocessing needed before model training.

Difficulty: INTERMEDIATE

 Inside this notebook, the main preprocessing steps are: validating that the datasets exist, building training and test dataloaders, inspecting one sample batch, and adapting the image format to each model family. The custom CNN can consume small CIFAR-style tensors directly, while the pretrained models additionally require resizing to `224 x 224` and ImageNet normalization to match their pretrained feature space.


In [ ]:
import os
import random
import time

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from torchvision.models import ResNet18_Weights, Swin_T_Weights, resnet18, swin_t
from torchvision.transforms.functional import pil_to_tensor


### Environment Setup

The next cell fixes the random seed and selects the best available device in the order `MPS -> CUDA -> CPU`. Keeping this logic near the top improves reproducibility and makes the hardware choice explicit.


In [ ]:
seed = 42
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)


### Dataset Loading
We are downloading the dataset from pytorch and filtering / relabeling it.
We also:
- convert images to tensors to use them as model input
- normalizing rgb values (from 0-255 to 0-1) so the model can process the images better. it also mitigates overflows


In [ ]:
car_label = 1
truck_label = 9


class FilteredCIFAR10(torch.utils.data.Dataset):
    def __init__(self, train=True):
        self.dataset = CIFAR10(root="./data", train=train, download=True)
        self.indices = [
            i for i, label in enumerate(self.dataset.targets)
            if label in (car_label, truck_label)
        ]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, original_label = self.dataset[self.indices[idx]]
        image = pil_to_tensor(image).float() / 255.0
        label = 0 if original_label == car_label else 1
        return image, label


if "train_dataset" not in globals() or "test_dataset" not in globals():
    train_dataset = FilteredCIFAR10(train=True)
    test_dataset = FilteredCIFAR10(train=False)
    print("Loaded filtered CIFAR-10 for local debugging.")
else:
    print("Using externally provided train_dataset and test_dataset.")


### Dataset Check and Hyperparameters

The following cell validates the expected dataset variables, builds dataloaders, and defines the training hyperparameters used throughout the notebook. These values should be reported because they are part of the modeling design, not just implementation details.


In [ ]:
if "train_dataset" not in globals() or "test_dataset" not in globals():
    raise ValueError("Please define train_dataset and test_dataset before running this notebook.")

batch_size = 64
num_workers = 0
cnn_epochs = 10
resnet_epochs = 5
swin_epochs = 5
cnn_learning_rate = 1e-3
resnet_learning_rate = 1e-4
swin_learning_rate = 1e-4
save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))
print("Batch size:", batch_size)
print("Epochs -> CNN:", cnn_epochs, "ResNet18:", resnet_epochs, "Swin-Tiny:", swin_epochs)
print("Learning rates -> CNN:", cnn_learning_rate, "ResNet18:", resnet_learning_rate, "Swin-Tiny:", swin_learning_rate)
print("Loss function: CrossEntropyLoss")
print("Optimizer family: Adam")


### Sample Batch Inspection

A quick batch visualization confirms that the tensors, labels, and dataloader output look correct before training starts. This is the light-weight exploration needed in this modeling-focused notebook.


In [ ]:
images, labels = next(iter(train_loader))
display_images = images.float()
if display_images.max() > 1:
    display_images = display_images / 255.0

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Unique labels in this batch:", torch.unique(labels))

plt.figure(figsize=(10, 3))
for i in range(min(6, len(display_images))):
    plt.subplot(1, 6, i + 1)
    image = display_images[i].permute(1, 2, 0).cpu().clamp(0, 1)
    plt.imshow(image)
    plt.title(f"label={labels[i].item()}")
    plt.axis("off")
plt.tight_layout()
plt.show()


## 4. Baseline Model

## Contribution Metadata
Author(s): Amirmasoud Ahmadi Kolsaraki  
Section: Baseline Model  
Method: [simple-cnn, supervised-learning]  
SHORT Description of contribution: Implemented and justified the baseline convolutional network.

Difficulty: INTERMEDIATE

The baseline is a compact CNN trained from scratch. It provides a fair comparison point because it uses standard convolution, pooling, and fully connected layers without relying on pretrained external features. The model is intentionally simple: if the transfer-learning models improve on it, the gain can be attributed to stronger learned visual representations rather than extra task-specific engineering.


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


### Shared Training Utilities

All models use the same training and evaluation structure so the comparison stays consistent. The helper functions below also make explicit why the pretrained models need different input preparation from the baseline CNN.


In [ ]:
imagenet_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
imagenet_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def prepare_batch(images, model_type="cnn"):
    images = images.to(device).float()
    if images.max() > 1:
        images = images / 255.0

    if model_type in {"resnet", "swin"}:
        images = F.interpolate(images, size=(224, 224), mode="bilinear", align_corners=False)
        mean = imagenet_mean.to(images.device)
        std = imagenet_std.to(images.device)
        images = (images - mean) / std

    return images


def evaluate_model(model, dataloader, criterion, model_type="cnn"):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = prepare_batch(images, model_type=model_type)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * labels.size(0)
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    loss_value = running_loss / total
    accuracy_value = correct / total
    return loss_value, accuracy_value


def train_model(model, train_loader, test_loader, criterion, optimizer, epochs, model_type="cnn"):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "test_loss": [],
        "test_accuracy": [],
    }

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        start_time = time.time()

        for images, labels in train_loader:
            images = prepare_batch(images, model_type=model_type)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_accuracy = correct / total
        test_loss, test_accuracy = evaluate_model(model, test_loader, criterion, model_type=model_type)
        epoch_time = time.time() - start_time

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["test_loss"].append(test_loss)
        history["test_accuracy"].append(test_accuracy)

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"train loss: {train_loss:.4f} | "
            f"train acc: {train_accuracy:.4f} | "
            f"test loss: {test_loss:.4f} | "
            f"test acc: {test_accuracy:.4f} | "
            f"time: {epoch_time:.1f}s"
        )

    return history


def plot_history(history, title):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="Train loss")
    plt.plot(epochs, history["test_loss"], label="Test loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title} Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_accuracy"], label="Train accuracy")
    plt.plot(epochs, history["test_accuracy"], label="Test accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{title} Accuracy")
    plt.legend()

    plt.tight_layout()
    plt.show()


def summarize_history(name, history):
    return {
        "model": name,
        "final_train_loss": history["train_loss"][-1],
        "final_train_accuracy": history["train_accuracy"][-1],
        "final_test_loss": history["test_loss"][-1],
        "final_test_accuracy": history["test_accuracy"][-1],
    }


### Baseline Training

The baseline CNN is trained with Adam and cross-entropy loss. The resulting curves and summary metrics establish the reference level that the proposed models should improve upon.


In [ ]:
simple_cnn = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
cnn_optimizer = torch.optim.Adam(simple_cnn.parameters(), lr=cnn_learning_rate)

cnn_history = train_model(
    model=simple_cnn,
    train_loader=train_loader,
    test_loader=test_loader,
    criterion=criterion,
    optimizer=cnn_optimizer,
    epochs=cnn_epochs,
    model_type="cnn",
)

plot_history(cnn_history, "Simple CNN")

cnn_path = os.path.join(save_dir, "simple_cnn_car_truck.pth")
torch.save(simple_cnn.state_dict(), cnn_path)
print("Saved baseline CNN to:", cnn_path)
print(summarize_history("SimpleCNN", cnn_history))


## 5. Proposed Recommendation Model

## Contribution Metadata
Author(s): Amirmasoud Ahmadi Kolsaraki  
Section: Proposed Recommendation Model  
Method: [transfer-learning, resnet18, swin-tiny]  
SHORT Description of contribution: Implemented two proposed transfer-learning models and justified their design choices.

Difficulty: ADVANCED

Here the proposed approaches are two pretrained vision backbones fine-tuned for the `car` vs `truck` task: `ResNet18` and `Swin-Tiny`. Both replace the final classification head with a two-logit output layer and fine-tune all parameters instead of freezing the backbone. This keeps the comparison aligned while testing whether a pretrained CNN or a pretrained vision transformer adapts better to the binary dataset.

Because both backbones were originally trained on ImageNet, the input pipeline resizes images to `224 x 224` and applies ImageNet normalization before the forward pass. Those preprocessing steps are not needed for the simpler baseline CNN.


### Proposed Model A: ResNet18

`ResNet18` is a strong transfer-learning baseline because residual connections make optimization stable while still keeping the model compact enough for a class project. The final fully connected layer is replaced so the network predicts two classes instead of the original ImageNet categories.


In [ ]:
weights = ResNet18_Weights.DEFAULT
resnet_model = resnet18(weights=weights)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, 2)
resnet_model = resnet_model.to(device)

resnet_criterion = nn.CrossEntropyLoss()
resnet_optimizer = torch.optim.Adam(resnet_model.parameters(), lr=resnet_learning_rate)

resnet_history = train_model(
    model=resnet_model,
    train_loader=train_loader,
    test_loader=test_loader,
    criterion=resnet_criterion,
    optimizer=resnet_optimizer,
    epochs=resnet_epochs,
    model_type="resnet",
)

plot_history(resnet_history, "ResNet18")

resnet_path = os.path.join(save_dir, "resnet18_car_truck.pth")
torch.save(resnet_model.state_dict(), resnet_path)
print("Saved ResNet18 to:", resnet_path)
print(summarize_history("ResNet18", resnet_history))


### Proposed Model B: Swin-Tiny

`Swin-Tiny` adds a transformer-based alternative to the pretrained CNN. It uses shifted-window self-attention, which gives the project a second proposed architecture with a different inductive bias from `ResNet18`. The classification head is replaced with a two-class output layer so the fine-tuning setup remains parallel to the ResNet experiment.


In [ ]:
swin_weights = Swin_T_Weights.DEFAULT
swin_model = swin_t(weights=swin_weights)
swin_model.head = nn.Linear(swin_model.head.in_features, 2)
swin_model = swin_model.to(device)

swin_criterion = nn.CrossEntropyLoss()
swin_optimizer = torch.optim.Adam(swin_model.parameters(), lr=swin_learning_rate)

swin_history = train_model(
    model=swin_model,
    train_loader=train_loader,
    test_loader=test_loader,
    criterion=swin_criterion,
    optimizer=swin_optimizer,
    epochs=swin_epochs,
    model_type="swin",
)

plot_history(swin_history, "Swin-Tiny")

swin_path = os.path.join(save_dir, "swin_tiny_car_truck.pth")
torch.save(swin_model.state_dict(), swin_path)
print("Saved Swin-Tiny to:", swin_path)
print(summarize_history("Swin-Tiny", swin_history))


### Current Comparison Snapshot

The next cell prints the final metrics for the three trained models. This is enough to support the modeling section now, while a later version of the notebook can expand these outputs into the template's dedicated evaluation, results, and discussion sections.


In [ ]:
results = [
    summarize_history("SimpleCNN", cnn_history),
    summarize_history("ResNet18", resnet_history),
    summarize_history("Swin-Tiny", swin_history),
]

for row in results:
    print(row)


### Reproducibility Note

The notebook saves all trained weights under `saved_models/` and includes an example loading cell below. This is useful later when the full report reaches the reproducibility section, but it is already included here so the modeling outputs can be reused by teammates without retraining.


In [ ]:
loaded_cnn = SimpleCNN()
loaded_cnn.load_state_dict(torch.load(cnn_path, map_location="cpu"))
loaded_cnn.eval()

loaded_resnet = resnet18(weights=None)
loaded_resnet.fc = nn.Linear(loaded_resnet.fc.in_features, 2)
loaded_resnet.load_state_dict(torch.load(resnet_path, map_location="cpu"))
loaded_resnet.eval()

loaded_swin = swin_t(weights=None)
loaded_swin.head = nn.Linear(loaded_swin.head.in_features, 2)
loaded_swin.load_state_dict(torch.load(swin_path, map_location="cpu"))
loaded_swin.eval()

print("All saved models were loaded successfully.")
